In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)
from sklearn.decomposition import PCA

In [2]:
# ─────────────────────────────────────────────
# 1. Load & Prepare Data
# ─────────────────────────────────────────────
df = pd.read_csv('../encoded_dataset.csv')
 
# Map target variable
# 5–7 → Procrastinate (1), 1–4 → Not Procrastinate (0)
df['procrastinate'] = df['delay_until_deadline'].apply(
    lambda x: 1 if x >= 5 else 0
)
 
# Top 15 features from the Random Forest importance chart
top15_features = [
    'delay_unenjoyable_assignments',
    'comfort_or_convenience',
    'far_away_deadlines',
    'self_study_hours',
    'control_over_habit',
    'impact_on_learning_progress',
    'leisure_hours',
    'difficult_to_maintain',
    'start_early_when_necessary',
    'peers_consider_normal',
    'study_plan',
    'focus_frequency',
    'sleep_hours',
    'sooner_next_month',
    'college',
]
 
X = df[top15_features].dropna()
y = df.loc[X.index, 'procrastinate']

In [3]:
# ─────────────────────────────────────────────
# 2. Fixed Split: 100 train / 94 test
# ─────────────────────────────────────────────
np.random.seed(42)
indices = np.random.permutation(len(X))
train_idx = indices[:100]
test_idx  = indices[100:194]
 
X_train = X.iloc[train_idx].values
X_test  = X.iloc[test_idx].values
y_train = y.iloc[train_idx].values
y_test  = y.iloc[test_idx].values
 
# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

In [4]:
# ─────────────────────────────────────────────
# 3. Train SVM
# ─────────────────────────────────────────────
svm = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
svm.fit(X_train_sc, y_train)
 
y_train_pred = svm.predict(X_train_sc)
y_test_pred  = svm.predict(X_test_sc)
y_test_prob  = svm.predict_proba(X_test_sc)[:, 1]

In [8]:
# ─────────────────────────────────────────────
# 4. Metrics
# ─────────────────────────────────────────────
def get_metrics(y_true, y_pred, label=""):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    print(f"\n{'─'*40}")
    print(f"  {label} Metrics")
    print(f"{'─'*40}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1-Score  : {f1:.4f}")
    return acc, prec, rec, f1
 
train_metrics = get_metrics(y_train, y_train_pred, "Training Set")
test_metrics  = get_metrics(y_test,  y_test_pred,  "Test Set")
 



────────────────────────────────────────
  Training Set Metrics
────────────────────────────────────────
  Accuracy  : 0.9100
  Precision : 0.9038
  Recall    : 0.9216
  F1-Score  : 0.9126

────────────────────────────────────────
  Test Set Metrics
────────────────────────────────────────
  Accuracy  : 0.6989
  Precision : 0.6757
  Recall    : 0.6098
  F1-Score  : 0.6410


In [9]:
print("\nClassification Report (Train Set):")
print(classification_report(y_train, y_train_pred,
      target_names=['Not Procrastinate', 'Procrastinate']))

print("\nClassification Report (Test Set):")
print(classification_report(y_test, y_test_pred,
      target_names=['Not Procrastinate', 'Procrastinate']))


Classification Report (Train Set):
                   precision    recall  f1-score   support

Not Procrastinate       0.92      0.90      0.91        49
    Procrastinate       0.90      0.92      0.91        51

         accuracy                           0.91       100
        macro avg       0.91      0.91      0.91       100
     weighted avg       0.91      0.91      0.91       100


Classification Report (Test Set):
                   precision    recall  f1-score   support

Not Procrastinate       0.71      0.77      0.74        52
    Procrastinate       0.68      0.61      0.64        41

         accuracy                           0.70        93
        macro avg       0.69      0.69      0.69        93
     weighted avg       0.70      0.70      0.70        93

